In [ ]:
%matplotlib widget

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# BESSEL-THOMSON LOW-PASS FILTER EXERCISE
#
# Filter order:
#
#       N = 8
#
# The notebook constructs the Bessel filter directly from the theoretical
# coefficient equation:
#
#       αk = (2N-k)! / [2^(N-k) k! (N-k)!]
#
# Therefore:
#
# 1. Generate all Bessel-polynomial coefficients symbolically.
# 2. Construct H(s) with H(0) = 1.
# 3. Calculate the poles from the generated polynomial.
# 4. Construct the conjugate-pole second-order factors.
# 5. Substitute s = jω to obtain H(jω).
# 6. Derive magnitude response.
# 7. Derive phase response.
# 8. Derive group delay.
#
# No coefficient, pole, or result from the printed solution is hard-coded.
# ==============================================================================

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}
</style>
"""))

# ==============================================================================
# FILTER ORDER
# ==============================================================================

N = 8

# ==============================================================================
# SYMBOLIC VARIABLES
# ==============================================================================

s = sp.symbols('s', real=True)
omega = sp.symbols('omega', real=True)
I = sp.I

# ==============================================================================
# STEP 1: BESSEL-POLYNOMIAL COEFFICIENTS
#
# αk = (2N-k)! / [2^(N-k) k! (N-k)!]
#
# k = 0,1,...,N
# ==============================================================================

alpha = []

for k in range(N + 1):
    alpha_k = sp.factorial(2 * N - k) / (2**(N - k) * sp.factorial(k) * sp.factorial(N - k))
    alpha.append(sp.simplify(alpha_k))

# ==============================================================================
# STEP 2: SYMBOLIC DENOMINATOR
#
# D(s) = Σ αk s^k
# ==============================================================================

denominator_symbolic = sp.Integer(0)

for k in range(N + 1):
    denominator_symbolic += alpha[k] * s**k

denominator_symbolic = sp.expand(denominator_symbolic)

# ==============================================================================
# STEP 3: NORMALIZATION CONSTANT
#
# H(0) = 1
#
# Therefore:
#
# β0 = α0 = D(0)
# ==============================================================================

beta0_symbolic = sp.simplify(denominator_symbolic.subs(s, 0))

H_s = sp.cancel(beta0_symbolic / denominator_symbolic)

# ==============================================================================
# STEP 4: POLES
#
# The poles are calculated numerically from the symbolically generated
# denominator polynomial.
# ==============================================================================

poles_symbolic = sp.nroots(denominator_symbolic, n=15, maxsteps=200)

poles_numeric = np.array([complex(p) for p in poles_symbolic])

poles_numeric = poles_numeric[np.argsort(-poles_numeric.imag)]

# ==============================================================================
# STEP 5: CONJUGATE-POLE FACTORS
# ==============================================================================

upper_poles = [p for p in poles_numeric if p.imag > 1e-10]

pole_factors = []

for p_k in upper_poles:
    sigma_k = p_k.real
    omega_k = p_k.imag
    factor_k = sp.expand(s**2 - 2.0 * sp.Float(sigma_k, 16) * s + sp.Float(sigma_k**2 + omega_k**2, 16))
    pole_factors.append(sp.N(factor_k, 10))

# ==============================================================================
# STEP 6: SYMBOLIC FREQUENCY RESPONSE
#
# s -> jω
# ==============================================================================

denominator_jw = sp.expand(denominator_symbolic.subs(s, I * omega))

den_real = sp.expand(sp.re(denominator_jw))
den_imag = sp.expand(sp.im(denominator_jw))

H_jw = sp.cancel(beta0_symbolic / denominator_jw)

# ==============================================================================
# STEP 7: SYMBOLIC MAGNITUDE RESPONSE
#
# |H(jω)| = β0 / sqrt[R²(ω) + I²(ω)]
# ==============================================================================

magnitude_denominator = sp.expand(den_real**2 + den_imag**2)

magnitude_squared_symbolic = sp.cancel(beta0_symbolic**2 / magnitude_denominator)

magnitude_symbolic = sp.sqrt(magnitude_squared_symbolic)

# ==============================================================================
# STEP 8: SYMBOLIC PHASE RESPONSE
#
# Since β0 > 0:
#
# ∠H(jω) = -atan2[I(ω), R(ω)]
#
# The analytical ratio appearing inside tan^-1 is:
#
# I(ω) / R(ω)
# ==============================================================================

phase_ratio_symbolic = sp.cancel(den_imag / den_real)

# ==============================================================================
# STEP 9: SYMBOLIC GROUP DELAY
#
# D(jω) = R(ω) + jI(ω)
#
# τ(ω) = [R I' - I R'] / [R² + I²]
# ==============================================================================

den_real_derivative = sp.diff(den_real, omega)
den_imag_derivative = sp.diff(den_imag, omega)

group_delay_numerator = sp.expand(den_real * den_imag_derivative - den_imag * den_real_derivative)

group_delay_denominator = sp.expand(den_real**2 + den_imag**2)

group_delay_symbolic = sp.cancel(group_delay_numerator / group_delay_denominator)

# ==============================================================================
# NUMERICAL FUNCTIONS FOR PLOTTING
# ==============================================================================

magnitude_function = sp.lambdify(omega, magnitude_symbolic, 'numpy')
den_real_function = sp.lambdify(omega, den_real, 'numpy')
den_imag_function = sp.lambdify(omega, den_imag, 'numpy')
group_delay_function = sp.lambdify(omega, group_delay_symbolic, 'numpy')

# ==============================================================================
# FREQUENCY AXIS
# ==============================================================================

omega_values = np.logspace(-2, 2, 5000)

# ==============================================================================
# MAGNITUDE VALUES
# ==============================================================================

magnitude_values = np.asarray(magnitude_function(omega_values), dtype=float)

# ==============================================================================
# PHASE VALUES
# ==============================================================================

den_real_values = np.asarray(den_real_function(omega_values), dtype=float)
den_imag_values = np.asarray(den_imag_function(omega_values), dtype=float)

phase_values = -np.unwrap(np.arctan2(den_imag_values, den_real_values))

phase_deg_values = np.rad2deg(phase_values)

# ==============================================================================
# GROUP-DELAY VALUES
# ==============================================================================

group_delay_values = np.asarray(group_delay_function(omega_values), dtype=float)

if group_delay_values.ndim == 0:
    group_delay_values = np.full_like(omega_values, float(group_delay_values))

# ==============================================================================
# DISPLAY TEXT: COEFFICIENTS
# ==============================================================================

coefficient_text = '<br>'.join([f'α{k} = {alpha[k]}' for k in range(N + 1)])

# ==============================================================================
# DISPLAY TEXT: POLES
# ==============================================================================

pole_text = '<br>'.join([f'p{k + 1} = {p.real:+.6f} {p.imag:+.6f}j' for k, p in enumerate(poles_numeric)])

# ==============================================================================
# DISPLAY TEXT: POLE FACTORS
# ==============================================================================

factor_text = ''

for index, factor_k in enumerate(pole_factors):
    factor_text += f'Pair {index + 1}: <span style="color:#0066cc;">{sp.sstr(factor_k)}</span><br>'

# ==============================================================================
# DISPLAY EXPRESSIONS
# ==============================================================================

denominator_display = sp.N(denominator_symbolic, 9)

den_real_display = sp.N(den_real, 9)

den_imag_display = sp.N(den_imag, 9)

magnitude_display = sp.N(magnitude_symbolic, 7)

phase_ratio_display = sp.N(phase_ratio_symbolic, 7)

group_delay_numerator_display = sp.N(group_delay_numerator, 7)

group_delay_denominator_display = sp.N(group_delay_denominator, 7)

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML(f"""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:9px 11px;
    margin:0px 0px 8px 0px;
    font-size:12px;
    line-height:1.50;
    background-color:#f7fbff;
    width:1220px;
    max-width:1220px;
    box-sizing:border-box;
">
<b>Bessel-Thomson Filter Exercise</b><br>
Design the amplitude and phase responses of a Bessel filter of order
N = {N}.
<br>
<b>Purpose:</b>
Construct the filter directly from the theoretical Bessel-polynomial
coefficient equation. The polynomial coefficients, transfer function, poles,
frequency response, magnitude response, phase response and group delay are
calculated by Python; no numerical result from the printed solution is
hard-coded.
</div>
""", layout=Layout(width='1230px', max_width='1230px'))

# ==============================================================================
# INFORMATION PANEL
# ==============================================================================

info_html = HTML(f"""
<div style="
    border:1px solid #cccccc;
    border-radius:7px;
    padding:10px 11px;
    font-size:12px;
    line-height:1.55;
    background:white;
    width:600px;
    box-sizing:border-box;
">

<b>Step 1 — Filter order</b><br>
N = <span style="color:#0066cc;"><b>{N}</b></span>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 2 — Bessel-polynomial coefficients</b><br>
<span style="color:#0066cc;">
{coefficient_text}
</span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 3 — Transfer function</b><br>
β₀ = <span style="color:#0066cc;">{beta0_symbolic}</span><br>
H(s) = β₀ / D(s)<br>
<span style="color:#0066cc;">
D(s) = {sp.sstr(denominator_display)}
</span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 4 — Poles</b><br>
<span style="color:#0066cc;">
{pole_text}
</span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 5 — Conjugate-pole factors</b><br>
{factor_text}
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 6 — Frequency response denominator</b><br>
D(jω) =
<span style="color:#0066cc;">
({sp.sstr(den_real_display)}) + j({sp.sstr(den_imag_display)})
</span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 7 — Magnitude response</b><br>
|H(jω)| =
<span style="color:#0066cc;">
{sp.sstr(magnitude_display)}
</span>
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 8 — Phase response</b><br>
∠H(jω) = −tan⁻¹[
<span style="color:#0066cc;">
{sp.sstr(phase_ratio_display)}
</span>
]
</div>

<div style="margin-top:8px; padding-top:7px; border-top:1px solid #eeeeee;">
<b>Step 9 — Group delay</b><br>
τ(ω) =
<span style="color:#0066cc;">
({sp.sstr(group_delay_numerator_display)})
/
({sp.sstr(group_delay_denominator_display)})
</span>
</div>

</div>
""", layout=Layout(width='610px', max_width='610px'))

# ==============================================================================
# COMMON FIGURE SETTINGS
# ==============================================================================

title_fontsize = 11
label_fontsize = 9
tick_fontsize = 8
legend_fontsize = 8

# ==============================================================================
# FIGURE 1: MAGNITUDE RESPONSE
# ==============================================================================

fig_mag, ax_mag = plt.subplots(figsize=(5.3, 3.0))

ax_mag.plot(omega_values, magnitude_values, 'r-', linewidth=2.0, label='|H(jω)|')

ax_mag.set_xscale('log')

ax_mag.set_xlabel('Angular Frequency ω (rad/s)', fontsize=label_fontsize)

ax_mag.set_ylabel('|H(jω)|', fontsize=label_fontsize)

ax_mag.set_title('Bessel Filter Magnitude Response', fontsize=title_fontsize, fontweight='bold', pad=5)

ax_mag.tick_params(axis='both', labelsize=tick_fontsize)

ax_mag.grid(True, which='both', linestyle=':', alpha=0.5)

ax_mag.legend(loc='upper center', bbox_to_anchor=(0.5, -0.23), ncol=1, fontsize=legend_fontsize)

ax_mag.set_xlim(0.01, 100.0)

ax_mag.set_ylim(0.0, 1.08)

fig_mag.subplots_adjust(left=0.14, right=0.97, bottom=0.30, top=0.85)

fig_mag.canvas.header_visible = False

fig_mag.canvas.toolbar_visible = False

fig_mag.canvas.resizable = False

fig_mag.canvas.layout.width = '530px'

fig_mag.canvas.layout.height = '305px'

# ==============================================================================
# FIGURE 2: PHASE RESPONSE
# ==============================================================================

fig_phase, ax_phase = plt.subplots(figsize=(5.3, 3.0))

ax_phase.plot(omega_values, phase_deg_values, 'r-', linewidth=2.0, label='∠H(jω)')

ax_phase.axhline(0.0, color='gray', linestyle='--', linewidth=0.8)

ax_phase.set_xscale('log')

ax_phase.set_xlabel('Angular Frequency ω (rad/s)', fontsize=label_fontsize)

ax_phase.set_ylabel('Phase (degrees)', fontsize=label_fontsize)

ax_phase.set_title('Bessel Filter Phase Response', fontsize=title_fontsize, fontweight='bold', pad=5)

ax_phase.tick_params(axis='both', labelsize=tick_fontsize)

ax_phase.grid(True, which='both', linestyle=':', alpha=0.5)

ax_phase.legend(loc='upper center', bbox_to_anchor=(0.5, -0.23), ncol=1, fontsize=legend_fontsize)

ax_phase.set_xlim(0.01, 100.0)

ax_phase.set_ylim(-725.0, 5.0)

ax_phase.set_yticks([0, -90, -180, -270, -360, -450, -540, -630, -720])

fig_phase.subplots_adjust(left=0.14, right=0.97, bottom=0.30, top=0.85)

fig_phase.canvas.header_visible = False

fig_phase.canvas.toolbar_visible = False

fig_phase.canvas.resizable = False

fig_phase.canvas.layout.width = '530px'

fig_phase.canvas.layout.height = '305px'

# ==============================================================================
# FIGURE 3: GROUP DELAY
# ==============================================================================

fig_gd, ax_gd = plt.subplots(figsize=(5.3, 3.0))

ax_gd.plot(omega_values, group_delay_values, 'r-', linewidth=2.0, label='τ(ω)')

ax_gd.axhline(0.0, color='gray', linestyle='--', linewidth=0.8)

ax_gd.set_xscale('log')

ax_gd.set_xlabel('Angular Frequency ω (rad/s)', fontsize=label_fontsize)

ax_gd.set_ylabel('Group Delay τ(ω)', fontsize=label_fontsize)

ax_gd.set_title('Bessel Filter Group Delay', fontsize=title_fontsize, fontweight='bold', pad=5)

ax_gd.tick_params(axis='both', labelsize=tick_fontsize)

ax_gd.grid(True, which='both', linestyle=':', alpha=0.5)

ax_gd.legend(loc='upper center', bbox_to_anchor=(0.5, -0.23), ncol=1, fontsize=legend_fontsize)

ax_gd.set_xlim(0.01, 100.0)

ax_gd.set_ylim(0.0, 1.15)

fig_gd.subplots_adjust(left=0.14, right=0.97, bottom=0.30, top=0.85)

fig_gd.canvas.header_visible = False

fig_gd.canvas.toolbar_visible = False

fig_gd.canvas.resizable = False

fig_gd.canvas.layout.width = '530px'

fig_gd.canvas.layout.height = '305px'

# ==============================================================================
# LAYOUT
# ==============================================================================

left_column = VBox([info_html], layout=Layout(width='620px', min_width='620px', max_width='620px', flex='0 0 620px', align_items='flex-start'))

right_column = VBox([fig_mag.canvas, fig_phase.canvas, fig_gd.canvas], layout=Layout(width='540px', min_width='540px', max_width='540px', flex='0 0 540px', align_items='flex-start'))

main_layout = HBox([left_column, right_column], layout=Layout(width='1170px', min_width='1170px', max_width='1170px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)

display(main_layout)